In [1]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
from sqlalchemy import text
import urllib
from databricks import sql
import os
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta

def connect_to_db(server, database):
    """
    Connect to SQL server: Parameters are specified in place
 
    """
    # TO DO: Specify connection parameters
    params = urllib.parse.quote_plus(
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={server};"
        f"DATABASE={database};"
        "Trusted_Connection=yes"
    )
 
    engine = sa.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params))
    return engine


# DATABRICKS CONNECTION 
# os.environ["DATABRICKS_SERVER_HOSTNAME"] = "dbc-1c3bb27b-2d8f.cloud.databricks.com"
# os.environ["DATABRICKS_HTTP_PATH"] = "/sql/1.0/warehouses/25e39a028cd7f5f5"


# databricks_connection = sql.connect(server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
#                           http_path=os.getenv("DATABRICKS_HTTP_PATH"),
#                           auth_type="databricks-oauth")

In [2]:
def last_valid_date(date_str):
    # Function that checks if date is valid and if not it returns the last valid date before specified date string.
    # Example last_valid_date('2024-02-31') = '2024-02-29'

    input_dt = date_str
    valid = False
    while not valid:
        try:
            date_obj = datetime.strptime(input_dt, '%Y-%m-%d')
            valid = True
        except:
            input_dt = input_dt[:7] + "-" + str(int(input_dt[-2:]) - 1)

    return input_dt

def get_lazy_payers(df, perimeter):

    # This functions only checks the last criterion from lazy payers. E.g. at most 3 months in bucket 1 in last 6 months
    # The other requirements from lazy payers (collection and max bucket=2) must be applied before this function
    df["nbd"] = df['ReferenceDate'].astype(str).str[:7] + "-" + df["NEXT_BILLING_DATE"].astype(str).str[-2:]
    df["nbd"] = df["nbd"].apply(last_valid_date)
    df["diff"] = abs((pd.to_datetime(df["nbd"]) - pd.to_datetime(df["ReferenceDate"])).dt.days)
    df = df.drop(columns="NEXT_BILLING_DATE")
    df = df[df.groupby(["AccountNumber", "DebtorCDI", "nbd"])["diff"].transform("min") == df["diff"]]
    # if for the same nbd we take the one corresponding to max reference date
    df = df[df.groupby(["AccountNumber", "DebtorCDI", "nbd", "diff"])["ReferenceDate"].transform("max") == df["ReferenceDate"]]
    # keep last 6 months
    df = df.groupby(["AccountNumber", "DebtorCDI"]).head(6)

    grouped_counts = df[df["SourceBucket"] == 2].groupby(["AccountNumber", "DebtorCDI"]).size() 
    # at most 2 months in bucket 2
    lp_ind = grouped_counts[grouped_counts <= 2].reset_index()
    lazy_payers = perimeter.merge(lp_ind.drop(columns=0), how="inner", on=["AccountNumber", "DebtorCDI"])
    
    return lazy_payers

In [6]:
def third_party_requirements(perimeter):
    # 3rd party perimeter requirements - from excel
    perimeter = perimeter[perimeter["FinalBucket"].isin([1,2,3])].reset_index(drop=True)
    perimeter = perimeter[perimeter["AssignmentProcess"].notnull()].reset_index(drop=True)
    perimeter["CustomerSegment_lower"] = perimeter["CustomerSegment"].apply(lambda x: x if x is None else x.lower())
    perimeter["AssignmentProcess_lower"] = perimeter["AssignmentProcess"].apply(lambda x: x if x is None else x.lower())

    # separate condition for Mercury and the other portfolios (Orbit, Light, Cell, Pearl A)
    merc = perimeter[perimeter["PortfolioName"] == "Mercury"].reset_index(drop=True)
    cond1 =  merc["AssignmentProcess"].isin(['3869 CA Dialer', 'CA Dialer', 'CA/FA/Sme Officer', 'Collection', 'Collection_Judicial_Escalation', 'DNC_Offer', 'Lf_Offer'])
    cond2 = ~merc["CustomerSegment_lower"].str.contains("non indexed|rest")
    merc = merc[cond1 & cond2]

    other = perimeter[perimeter["PortfolioName"] != "Mercury"].reset_index(drop=True)
    other = other[other["AssignmentProcess_lower"].str.contains("collection")].reset_index(drop=True)

    perimeter = pd.concat([merc, other]).reset_index(drop=True)
    perimeter = perimeter.drop(columns=["CustomerSegment_lower", "AssignmentProcess_lower"])
    return perimeter

In [7]:
# TO DO: the last reference date should be today - 1 (or 2 days clarify it!) kai na perftei se working day
def extract_perimeter(category, 
                      query_temp_perimeter_path, 
                      query_perimeter_path, 
                      query_temp_lazy_payers_path,
                      query_lazy_payers_path,
                      query_3rd_party_case_codes_path="Queries/3rd party/query_3rd_party_case_codes", 
                      third_party_exlusion_queues_path="3rd parties_Exclusion Queues.xlsx"):
    
    # category should be one of ["Core", "Securitized", "3rd party"]
    queries = []
    for path in [query_temp_perimeter_path, query_perimeter_path, query_temp_lazy_payers_path, query_lazy_payers_path, query_3rd_party_case_codes_path]:
        with open(path, "r", encoding="utf-8") as f:
            queries.append(f.read())
    
    query_temp_perimeter, query_perimeter, query_temp_lazy_payers, query_lazy_payers, query_3rd_party_case_codes = queries
    
        
    if category == "Core":
        engine = connect_to_db(server="DWHDATAMARTS", database="Core")
        with engine.connect() as conn:
            conn.execute(text(query_temp_perimeter))
            conn.execute(text(query_perimeter))
            conn.execute(text(query_temp_lazy_payers))
            perimeter = pd.read_sql("select * from #core_perimeter", con=conn)
            perimeter = perimeter.drop_duplicates().reset_index(drop=True)
            lazy_payers = pd.read_sql(query_lazy_payers, con=conn)
            core_collections = perimeter[perimeter["CourseOfAction"].isin(["Collection", "collection"])].reset_index(drop=True)
            lazy_payers = get_lazy_payers(lazy_payers, core_collections)
    elif category == "Securitized":
        dfs, lazy_payers = [], []
        for portfolio in ["GalaxyRetail", "CosmosRetail"]:
            engine = connect_to_db(server="DWHDATAMARTS", database=portfolio)
            with engine.connect() as conn:
                conn.execute(text(query_temp_perimeter))
                conn.execute(text(query_perimeter))
                conn.execute(text(query_temp_lazy_payers))
                df = pd.read_sql("select * from #sec_perimeter", con=conn)
                lp = pd.read_sql(query_lazy_payers, con=conn)
            
            dfs.append(df)
            lazy_payers.append(lp)
        perimeter = pd.concat(dfs).reset_index(drop=True)
        lazy_payers = pd.concat(lazy_payers).reset_index(drop=True)
        perimeter = perimeter.drop_duplicates().reset_index(drop=True)
        sec_collections = perimeter[perimeter["CourseOfAction"].isin(["Collection", "collection"])].reset_index(drop=True)
        lazy_payers = get_lazy_payers(lazy_payers, sec_collections)
    else:
        engine = connect_to_db(server="DWHCORE", database="Analytic")
        with engine.connect() as conn:
            conn.execute(text(query_temp_perimeter))
            perimeter = pd.read_sql(query_perimeter, con=conn)

        exclusion_queues = pd.read_excel(third_party_exlusion_queues_path)
        perimeter = perimeter[~perimeter["Queue"].isin(exclusion_queues["queue"])]

        # apply 3rd party perimeter requirements
        perimeter = third_party_requirements(perimeter)

        # Find case codes from DWH_STG
        engine = connect_to_db(server="DWHCORE", database="DWH_STG")
        with engine.connect() as conn:
            case_codes = pd.read_sql(query_3rd_party_case_codes, con=conn)

        perimeter["AccountNumber_new"] = perimeter["AccountNumber"].apply(lambda x: x[3:] if x[:3].isalpha() else x)
        perimeter = perimeter.merge(case_codes, how="left", left_on=["AccountNumber_new", "AgreementNumber"], right_on=["Account", "contract_number"])
        perimeter = perimeter.drop(columns=["Account", "AccountNumber_new", "contract_number"])
        perimeter = perimeter.drop_duplicates().reset_index(drop=True)

        # compute lazy payers for 3rd party
        # we connect to datamarts to bring the billing date because this information does not exist in DWHCORE
        lazy_payers = []
        # pearl database is empty (17/03/2024)
        for portfolio in ["MERCURY", "Orbit", "Light", "CELL"]:
            engine = connect_to_db(server="DWHDATAMARTS", database=portfolio)
            with engine.connect() as conn:
                conn.execute(text(query_temp_lazy_payers))
                lp = pd.read_sql(query_lazy_payers, con=conn)
                lazy_payers.append(lp)
        lazy_payers = pd.concat(lazy_payers).reset_index(drop=True)
        lazy_payers = get_lazy_payers(lazy_payers, perimeter)

    return perimeter, lazy_payers

In [47]:
core_perimeter, lazy_payers_core = extract_perimeter("Core",
                                    "Queries/Core/query_temp_core_perimeter",
                                    "Queries/Core/query_core_perimeter",
                                    "Queries/Core/query_temp_core_lazy_payers",
                                    "Queries/Core/query_core_lazy_payers"
                                    )

In [48]:
securitized_perimeter, lazy_payers_sec = extract_perimeter("Securitized",
                                                     "Queries/Securitized/query_temp_sec_perimeter",
                                                     "Queries/Securitized/query_sec_perimeter",
                                                     "Queries/Securitized/query_temp_sec_lazy_payers",
                                                     "Queries/Securitized/query_sec_lazy_payers"
                                                     )

In [8]:
third_party_perimeter, lazy_payers_3rd_party = extract_perimeter("3rd party", 
                                                                "Queries/3rd party/query_temp_3rd_party_perimeter",
                                                                "Queries/3rd party/query_3rd_party_perimeter",
                                                                "Queries/3rd party/query_temp_3rd_party_lazy_payers",
                                                                "Queries/3rd party/query_3rd_party_lazy_payers")


In [50]:
export_columns = ["Criteria Name", "Account Number", "Case Code", "CIF/GROUP", "AFM", "Relation Type", "Queue", "Contract Number", "Application Number", "DaysPastDue", "Bucket", "Deceased flag",
                  "Business Unit", "Legal Entity", "Product", "SubProduct", "Portfolio", "SPV", "Dispute Activity", "Mobile Phone", "Mobile Phone Active", "Do not call", "Is Dialer Phone", "Mobile Phone work flag",
                  "Mobile Phone Source", "RPC last 2 days", "Text body", "Segment", "Assignment Channel", "ReferenceDate"]

renamed_cols = {"AccountNumber": "Account Number", "DebtorCDI": "CIF/GROUP", "PrimaryDebtorCDI": "CIF/GROUP", "TaxNumber": "AFM", 'PortfolioName': "Portfolio",
                   "SourceBucket": "Bucket", "SourcePastDueDays": "DaysPastDue", "AgreementNumber": "Contract Number", "Queues": "Queue",
                   "RelationType": "Relation Type", "CaseCode": "Case Code", "AssetSegment": "Business Unit", "LegalForm": "Legal Entity", "ProductDescription": "Product",
                   'SubProductDescription': "SubProduct", "ProductDesc": "Product", "SubProductDesc": "SubProduct", "TxtEntFinalSegment": "Segment", "FinalSegment": "Segment", "CustomerSegment": "Segment",
                   "AssignmentChannel": "Assignment Channel", "Channel": "Assignment Channel", 'DeceasedFlag': "Deceased flag"
                }


def keep_columns(df, renamed_cols, export_columns):
    # rename columns, keep only these that exist in export and reorder them 
    df = df.rename(columns=renamed_cols)
    keep_cols = [col for col in df.columns if col in export_columns]
    df = df[keep_cols]
    reorder_cols = [col for col in export_columns if col in df.columns]
    df = df[reorder_cols]
    return df

#### 1. Payers to receive an informative sms

In [51]:
# 1. Lazy payers - Bucket 0

def find_lazy_payers_B0(core_perimeter, securitized_perimeter, third_party_perimeter, renamed_cols, export_columns):
    lazy_payers_B0 = []
    for df_perimeter in [core_perimeter, securitized_perimeter, third_party_perimeter]:
        cond = (df_perimeter["SourcePastDueDays"] == 0) & (df_perimeter["SourceBucket"] == 0)
        lp = df_perimeter[cond].reset_index(drop=True)
        lp["Criteria Name"] = "Lazy payers - Bucket 0"
        lp = keep_columns(lp, renamed_cols, export_columns)
        lazy_payers_B0.append(lp)

    lazy_payers_B0 = pd.concat(lazy_payers_B0).reset_index(drop=True)

    return lazy_payers_B0


#### 2. Lazy Payers

In [52]:
def find_lazy_payers(lazy_payers_core, lazy_payers_sec, lazy_payers_3rd_party, renamed_cols, export_columns):
    # TO DO: do the rename before concat
    lazy_payers = []
    for lp in [lazy_payers_core, lazy_payers_sec, lazy_payers_3rd_party]:
        lp = keep_columns(lp, renamed_cols, export_columns)
        lazy_payers.append(lp)
    
    lazy_payers = pd.concat(lazy_payers).reset_index(drop=True)
    lazy_payers["Criteria Name"] = "Lazy payers"
    return lazy_payers


#### 3. Promise reminder

In [53]:
def find_promise_reminder(core_perimeter, renamed_cols, export_columns):
    # Only for Core portfolio
    today = datetime.now().date()
    previous_day = today - timedelta(days=1)

    # accounts assigned only for collection or double treatment
    promise_reminder = core_perimeter[core_perimeter["CourseOfAction"].isin(["Collection", "collection", "Double Treatment"])].reset_index(drop=True)
    # with promise status pending
    promise_reminder = promise_reminder[promise_reminder["PromiseStatus"].isin(["Pending", "Pending (Replaced)"])].reset_index(drop=True)
    # applicable from promisue due date
    promise_reminder["PromiseDueDate"] = pd.to_datetime(promise_reminder["PromiseDueDate"], errors="coerce")
    promise_reminder["ActivityDate"] = pd.to_datetime(promise_reminder["ActivityDate"], errors="coerce")
    promise_reminder = promise_reminder[promise_reminder["PromiseDueDate"].dt.date <= today].reset_index(drop=True)
    
    # no rpc the previous day
    cond1 =  promise_reminder["ActivityDate"].dt.date != previous_day
    cond2 = (promise_reminder["ActivityDate"].dt.date == previous_day) & (promise_reminder["TotalRPCs"] == 0)
    promise_reminder = promise_reminder[cond1|cond2].reset_index(drop=True)

    promise_reminder["Criteria Name"] = "Promise Reminder"
    promise_reminder = keep_columns(promise_reminder, renamed_cols, export_columns)

    return promise_reminder

#### 4. Inflow/Outflow

In [54]:
def find_inflow_outflow(core_perimeter, renamed_cols, export_columns):
    # Only for Core

    today = datetime.now().date()
    previous_day = today - timedelta(days=1)
    next_month = today + relativedelta(months=1)
    current_year = today.year

    core_collections = core_perimeter[core_perimeter["CourseOfAction"].isin(["Collection", "collection"])].reset_index(drop=True)
    # not saved 
    inflow_outflow = core_collections[core_collections["FinalSaved"] == 'Not Saved'].reset_index(drop=True)
    # No promise status pending or broken - 7 days before billing date. Here we know that promise status is the most recent according to activity date
    inflow_outflow = inflow_outflow[~inflow_outflow["PromiseStatus"].isin(['Pending', 'Pending (Replaced)', 'Broken', 'Broken (Replaced)'])].reset_index(drop=True)
    # no rpc the previous day
    cond1 =  inflow_outflow["ActivityDate"] != previous_day
    cond2 = (inflow_outflow["ActivityDate"] == previous_day) & (inflow_outflow["TotalRPCs"] == 0)
    inflow_outflow = inflow_outflow[cond1|cond2].reset_index(drop=True)
    
    # assigned for inflow/outflow
    #AssignmentSegmentDetailed like 'inflow' or 'outflow'
    inflow_outflow = inflow_outflow[inflow_outflow["AssignmentSegmentDetailed"].notnull()].reset_index(drop=True)
    inflow_outflow["AssignmentSegmentDetailed"] = inflow_outflow["AssignmentSegmentDetailed"].apply(lambda x: x.lower())
    inflow_outflow = inflow_outflow[inflow_outflow["AssignmentSegmentDetailed"].str.contains("inflow|outflow")].reset_index(drop=True)
    
    # inflow 
    inflow_outflow["InflowMonth"] = pd.to_datetime(inflow_outflow["InflowMonth"], format="%Y-%m")
    cond1_inflow = (inflow_outflow["InflowMonth"].dt.year == today.year) & (inflow_outflow["InflowMonth"].dt.month == today.month)
    cond2_inflow = inflow_outflow["InflowMonth"] == next_month.strftime("%Y-%m")
    inflow = inflow_outflow[cond1_inflow|cond2_inflow].reset_index(drop=True)
    
    # outflow
    cond1_outflow = (inflow_outflow["OperationalBucket"] < 4)
    cond2_outflow = (inflow_outflow["ExitMonthGroupedNumber"] == "Expired")
    inflow_outflow["ExitMonthGroupedNumber"] = pd.to_datetime(inflow_outflow["ExitMonthGroupedNumber"], format="%Y%m", errors="coerce")
    cond3_outflow = (inflow_outflow["ExitMonthGroupedNumber"] <= pd.to_datetime(next_month))
    outflow = inflow_outflow[cond1_outflow & (cond2_outflow | cond3_outflow)].reset_index(drop=True)

    final_inflow_outflow = pd.concat([inflow, outflow]).reset_index(drop=True)
    final_inflow_outflow["Criteria Name"] = "Inflow / Outflow"
    final_inflow_outflow = keep_columns(final_inflow_outflow, renamed_cols, export_columns)

    return final_inflow_outflow

#### 5. Debt reminder

In [55]:
def find_debt_reminder(core_perimeter, third_party_perimeter, lazy_payers_core, lazy_payers_3rd_party, renamed_cols, export_columns):
    # For Core and 3rd party
    today = datetime.now().date()
    last_10_day = today - timedelta(days=10)

    debt_reminder = []
    for category in ["Core", "3rd party"]:
        if category == "Core":
            # assigned for collection or double treatment
            df = core_perimeter[core_perimeter["CourseOfAction"].isin(["Collection", "collection", "Double Treatment"])].reset_index(drop=True)
            # not lazy payer
            lp_df = lazy_payers_core.rename(columns={"AccountNumber": "AccountNumber_lp", "DebtorCDI": "DebtorCDI_lp"})[["AccountNumber_lp", "DebtorCDI_lp"]]
            df = df.merge(lp_df, how="left", left_on=["AccountNumber", "DebtorCDI"], right_on=["AccountNumber_lp", "DebtorCDI_lp"], indicator=True)
            df = df[df["_merge"] == "left_only"].drop(columns=["_merge", "AccountNumber_lp", "DebtorCDI_lp"])
            # pending promise null or broken
            df = df[(df["PromiseStatus"].isnull()) | (df["PromiseStatus"].isin(['Broken' , 'Broken (Replaced)']))].reset_index(drop=True)
            # no rpc for 10 days
            cond1 = ((df["ActivityDate"] >= last_10_day) & (df["ActivityDate"] <= today) & (df["TotalRPCs"] == 0))
            cond2 = (df["ActivityDate"] < last_10_day)
            df = df[cond1|cond2].reset_index(drop=True)
            # risk bucket<=3 and not saved
            df = df[(df["RiskBucket"] <= 3) & (df["FinalSaved"] == "Not Saved")].reset_index(drop=True)
            df["Criteria Name"] = "Debt Reminder"
            df = keep_columns(df, renamed_cols, export_columns)
            debt_reminder.append(df)
        else:
            # category is 3rd party
            # not lazy payer
            lp_df = lazy_payers_3rd_party.rename(columns={"AccountNumber": "AccountNumber_lp", "DebtorCDI": "DebtorCDI_lp"})[["AccountNumber_lp", "DebtorCDI_lp"]]
            df = third_party_perimeter.merge(lp_df, how="left", left_on=["AccountNumber", "DebtorCDI"], right_on=["AccountNumber_lp", "DebtorCDI_lp"], indicator=True)
            df = df[df["_merge"] == "left_only"].drop(columns=["_merge", "AccountNumber_lp", "DebtorCDI_lp"])
            # pending promise null or broken
            df = df[(df["PromiseStatus"].isnull()) | (df["PromiseStatus"].isin(['Broken' , 'Broken (Replaced)']))].reset_index(drop=True)
            # no rpc for 10 days
            cond1 = ((df["ActivityDate"] >= last_10_day) & (df["ActivityDate"] <= today) & (df["ContactType"] != "RIGHT PARTY CONTACT"))
            cond2 = (df["ActivityDate"] < last_10_day)
            df = df[cond1|cond2].reset_index(drop=True)
            # dpd > 10 and dpd < 150
            df = df[(df["SourcePastDueDays"] > 10) & (df["SourcePastDueDays"] < 150)].reset_index(drop=True)
            df["Criteria Name"] = "Debt Reminder"
            df = keep_columns(df,renamed_cols, export_columns)
            debt_reminder.append(df)

    debt_reminder = pd.concat(debt_reminder).reset_index(drop=True)
    
    return debt_reminder

#### 6. Invitation to restructure

In [56]:
def find_invitation_to_restructure(core_perimeter, renamed_cols, export_columns):
    # Only for Core

    # assigned for double treatment
    double_treatment_core = core_perimeter[core_perimeter["CourseOfAction"] == "Double Treatment"].reset_index(drop=True)
    # not saved
    inv_to_res = double_treatment_core[double_treatment_core["FinalSaved"] == "Not Saved"].reset_index(drop=True)\
    # pending promise null or broken
    inv_to_res = inv_to_res[(inv_to_res["PromiseStatus"].isnull()) | (inv_to_res["PromiseStatus"].isin(['Broken' , 'Broken (Replaced)']))].reset_index(drop=True)
    # without any active application
    inv_to_res = inv_to_res[inv_to_res["ApplicationStatus"] != 'Ενεργή'].reset_index(drop=True)
    # borrowers with EBAStatus PE/FPE
    inv_to_res = inv_to_res[inv_to_res["EBAStatus"].isin(['PL', 'FPL'])].reset_index(drop=True)

    inv_to_res["Criteria Name"] = "Invitation to Restructure"
    inv_to_res = keep_columns(inv_to_res, renamed_cols, export_columns)
    
    return inv_to_res

#### 7. Follow up on partially or not fulfilled settlements

In [5]:
def find_follow_up(third_party_perimeter, renamed_cols, export_columns):  
    # Only for 3rd party Mercury portfolio
    today = datetime.now().date()
    last_10_day = today - timedelta(days=10)
    # only for mercury 
    third_party_perimeter = third_party_perimeter[third_party_perimeter["PortfolioName"] == "Mercury"].reset_index(drop=True)
    # last settlement status not fulfilled or partially fulfilled
    follow_up = third_party_perimeter[third_party_perimeter["SettlementStatus"].isin(['Not Fulfilled', 'Partially Fulfilled'])].reset_index(drop=True)
    # DPD > 10 and DPD < 150
    follow_up = follow_up[(follow_up["SourcePastDueDays"] > 10) & (follow_up["SourcePastDueDays"] < 150)].reset_index(drop=True)
    # without pending promise
    follow_up = follow_up[follow_up["PromiseStatus"] != "Pending"].reset_index(drop=True)
    # no rpc for 10 days
    cond1 = ((follow_up["ActivityDate"] >= last_10_day) & (follow_up["ActivityDate"] <= today) & (follow_up["ContactType"] != "RIGHT PARTY CONTACT"))
    cond2 = (follow_up["ActivityDate"] < last_10_day)
    follow_up = follow_up[cond1|cond2].reset_index(drop=True)

    follow_up["Criteria Name"] = "Follow up on partially or not fulfilled settlement"
    follow_up = keep_columns(follow_up, renamed_cols, export_columns)

    return follow_up

### Exports

In [58]:
def extract_phones(query_temp_phones_path, query_phones_path):
    
    queries = []
    for path in [query_temp_phones_path, query_phones_path]:
        with open(path, "r", encoding="utf-8") as f:
            queries.append(f.read())
    
    query_temp_phones, query_phones = queries
    engine = connect_to_db(server="DWHCORE", database="DWH_STG")
    with engine.connect() as conn:
        conn.execute(text(query_temp_phones))
        phones = pd.read_sql(query_phones, con=conn)

    return phones

def apply_exclusions(df, legal_form_path="legal forms.xlsx"):
    
    # customer should have a mobile phone
    df = df[df["Mobile Phone"].notnull()].reset_index(drop=True)
    
    # customer not deceased
    df = df[df["Deceased flag"] == False].reset_index(drop=True)
    
    # not in dispute activity
    df = df[df["Dispute Activity"] == 0].reset_index(drop=True)
    
    # exclude specified legal forms
    exclusion_legal_forms = pd.read_excel(legal_form_path)
    df = df[~df["Legal Entity"].isin(exclusion_legal_forms["LegalForm"])].reset_index(drop=True)

    # exclude specific phone numbers
    df =  df[~df["Mobile Phone"].isin(["6900000000", "6999999999"])].reset_index(drop=True)
    df = df[~df["Mobile Phone"].str.endswith("00000")].reset_index(drop=True)
    
    # active mobile phone
    df = df[df["Mobile Phone Active"] == True].reset_index(drop=True)
    
    # phone number is in dialer queue
    df = df[df["Is Dialer Phone"] == 1].reset_index(drop=True)
    
    # no rpc last 2 days
    df = df[df["RPC last 2 days"] == 0].reset_index(drop=True)
    
    # do not call
    df = df[df["Do not call"] == False].reset_index(drop=True)
    
    # work flag - not work phone
    df = df[df["Mobile Phone work flag"] == 0].reset_index(drop=True)
    
    # phone 1 to many - keep phones belonging to only one AFM
    df = df[df.groupby("Mobile Phone")["AFM"].transform('nunique') == 1].reset_index(drop=True)

    return df


def fix_column_types(df):
    for col in ["Case Code", "DaysPastDue", "Bucket", "Is Dialer Phone", "Mobile Phone work flag"]:
        df[col] = df[col].astype("Int64")

    for col in ["Deceased flag", "Mobile Phone Active", "Do not call"]:
        df[col] = df[col].apply(lambda x: x if x is None else int(x))

    return df

sms_texts = {"Lazy payers - Bucket 0": "text 1",
            "Lazy payers": "text 2",
            "Promise Reminder": "text 3",
            "Inflow / Outflow": "text 4",
            "Debt Reminder": "text 5",
            "Invitation to Restructure": "text 6",
            "Follow up on partially or not fulfilled settlement": "text 7"
             }

In [59]:
lazy_payers_B0 = find_lazy_payers_B0(core_perimeter, securitized_perimeter, third_party_perimeter, renamed_cols, export_columns)
lazy_payers = find_lazy_payers(lazy_payers_core, lazy_payers_sec, lazy_payers_3rd_party, renamed_cols, export_columns)
promise_reminder = find_promise_reminder(core_perimeter, renamed_cols, export_columns)
inflow_outflow = find_inflow_outflow(core_perimeter, renamed_cols, export_columns)
debt_reminder = find_debt_reminder(core_perimeter, third_party_perimeter, lazy_payers_core, lazy_payers_3rd_party, renamed_cols, export_columns)
invitation_to_restructure = find_invitation_to_restructure(core_perimeter, renamed_cols, export_columns)
follow_up = find_follow_up(third_party_perimeter, renamed_cols, export_columns)

criteria = pd.concat([lazy_payers_B0, lazy_payers, promise_reminder,  inflow_outflow, debt_reminder, invitation_to_restructure, follow_up]).reset_index(drop=True)

phones = extract_phones("Queries/query_temp_phones", "Queries/query_phones")
criteria = criteria.merge(phones, how="left", left_on="CIF/GROUP", right_on="CDI")
criteria = criteria.drop(columns=["CDI", "AFM_phones"])
criteria["Text body"] = criteria["Criteria Name"].map(sms_texts)
criteria["Application Number"] = None
criteria = criteria[export_columns] 

#### Business users export

In [60]:
today = datetime.now().date()
output = apply_exclusions(criteria)
output = fix_column_types(output)
output = output.drop_duplicates()
qualco_ab = output.loc[output["Portfolio"].isin(['Core Retail', 'Galaxy II Funding DAC', 'Orion X Securitisation DAC', 'Cosmos Retail']), ["Case Code"]]
qualco_sor = output.loc[output["Portfolio"].isin(['Light', 'Orbit', 'Mercury', 'Pearl Portfolio A', 'Cell']), ["Case Code"]]

output.to_csv(f"Exports/business_users_export_{today}.csv", index=False)
qualco_ab.to_csv(f"Exports/Qualco_AB_{today}.csv", index=False)
qualco_sor.to_csv(f"Exports/Qualco_Sor_{today}.csv", index=False)